# TNF-alpha binder design with BoltzGen (Colab T4, compute-capped)
Run top to bottom. **Runtime -> Change runtime type -> T4 GPU first.** Cells C-E are cheap; only F (pilot) and G (scale) use real GPU time.
Rules: no paid compute without your approval; stop whenever the GPU-hour cap in cell B is reached.

In [ ]:
# A. GPU + Python check (stops early if no GPU)
import subprocess, sys
print(sys.version)
out = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, 'No GPU: switch the runtime to T4 before continuing'
assert sys.version_info >= (3, 11), 'boltzgen 0.3.2 needs Python >= 3.11'

In [ ]:
# B. Settings (edit here only)
GPU_HOUR_CAP = 3.0        # total hours of GPU you are willing to spend on stage G
PILOT_DESIGNS = 24        # stage F size
FINAL_BUDGET = 40         # designs kept after BoltzGen filtering (we submit up to 20)
SPEC = 'tnfa_e3_core.yaml'  # or tnfa_e3_wide.yaml
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/tnfa_boltzgen'
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
# C. Install BoltzGen 0.3.2 (pinned) and fetch the repo (public)
!pip -q install boltzgen==0.3.2
!rm -rf /content/repo && git clone -q https://github.com/SyzenShen/adaptyv-binder-challenges /content/repo
%cd /content/repo/challenges/tnfa/boltz
!ls

In [ ]:
# D. Validate the spec on CPU before any GPU work
!boltzgen check $SPEC --cache $OUT/models

In [ ]:
# E. Download model weights once (cached on Drive so a reset does not re-download). Full output is shown.
import os; os.environ['HF_HOME'] = OUT + '/hf'
!boltzgen download all --cache $OUT/models
!ls $OUT/models | head

In [ ]:
import subprocess, sys, time

def run_logged(cmd, log):
    """Run a shell command, stream its output into the notebook and a log file on Drive, return the exit code."""
    t0 = time.time()
    with open(log, 'w') as fh:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            sys.stdout.write(line); fh.write(line)
        p.wait()
    return p.returncode, time.time() - t0

# F. PILOT: tiny run to measure seconds per design. Output is shown below and saved to Drive (pilot.log).
cmd = f'boltzgen run {SPEC} --output {OUT}/pilot --protocol protein-anything --num_designs {PILOT_DESIGNS} --budget 8 --inverse_fold_avoid C --cache {OUT}/models --reuse'
print(cmd)
rc, dt = run_logged(cmd, OUT + '/pilot.log')
print('return code', rc, 'seconds', round(dt))
if rc != 0:
    raise SystemExit('Pilot failed: copy the last 40 lines printed above (or pilot.log on Drive) and send them back.')
sec_per = dt / PILOT_DESIGNS
n_final = int(0.8 * GPU_HOUR_CAP * 3600 / sec_per)
print(f'{sec_per:.1f} s per design incl. all steps; with a {GPU_HOUR_CAP} h cap use num_designs about {n_final}')
open(OUT + '/pilot_timing.txt','w').write(f'pilot_designs={PILOT_DESIGNS}\nseconds={dt:.0f}\nsec_per_design={sec_per:.1f}\nsuggested_num_designs={n_final}\n')

In [ ]:
# G. SCALE: run only after reading the pilot line above. Edit N if you want fewer.
N = n_final
!boltzgen run $SPEC --output $OUT/main --protocol protein-anything --num_designs $N --budget $FINAL_BUDGET --inverse_fold_avoid C --cache $OUT/models --reuse
# Interrupted or reset? Reconnect to a T4, re-run A-E and this cell: --reuse continues from finished steps.

In [ ]:
# H. Package results for download / for the assistant (small files only)
!cd $OUT && zip -qr tnfa_results.zip pilot_timing.txt main/final_ranked_designs main/intermediate_designs_inverse_folded 2>/dev/null; ls -la $OUT/tnfa_results.zip
print('Upload tnfa_results.zip (or the main/final_ranked_designs folder) back to the session.')